# QASIC Digital Twin - Design Overview

This notebook provides an interactive overview of QASIC quantum ASIC designs. You can:
- Load and validate design files
- Visualize qubit topologies and layouts
- Inspect design parameters
- Run basic validation checks
- Export design summaries

**Author**: QASIC Hardware Team  
**Version**: 1.0  
**Date**: March 2026

In [ ]:
# Import required libraries
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
from datetime import datetime

# Add digital-twin to path
sys.path.append('../')

from digital_twin.orchestrator import DigitalTwinOrchestrator
from digital_twin.schemas.design_schema import validate_design_json

# Initialize orchestrator
orchestrator = DigitalTwinOrchestrator()

print("✅ Digital Twin Platform initialized")
print(f"📅 {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Design File Selection

Select a design file to load and analyze. You can use designs created with `qasic-create` or provide your own design.json file.

In [ ]:
# Design file selection widget
design_files = list(Path('../').glob('**/*design*.json'))
design_files.extend([Path('./sample_designs/8qubit_linear.json'), 
                     Path('./sample_designs/16qubit_grid.json')])

# Filter existing files
existing_files = [f for f in design_files if f.exists()]

if existing_files:
    file_selector = widgets.Dropdown(
        options=[str(f) for f in existing_files],
        description='Design File:',
        style={'description_width': 'initial'}
    )
else:
    file_selector = widgets.Text(
        value='../designs/design.json',
        description='Design File:',
        style={'description_width': 'initial'}
    )

load_button = widgets.Button(description='Load Design', button_style='primary')
output_area = widgets.Output()

# Global variables to store loaded design
current_design = None
current_design_path = None

def load_design(b):
    global current_design, current_design_path
    
    with output_area:
        output_area.clear_output()
        
        try:
            design_path = Path(file_selector.value)
            
            if not design_path.exists():
                print(f"❌ Design file not found: {design_path}")
                print("\n💡 Try creating a design first:")
                print("   qasic-create --template 8qubit_linear --output design.json")
                return
            
            # Load and validate design
            with open(design_path, 'r') as f:
                design_data = json.load(f)
            
            # Validate against schema
            validated_design = validate_design_json(design_data)
            
            current_design = validated_design
            current_design_path = design_path
            
            print(f"✅ Design loaded successfully: {design_path}")
            print(f"📊 Project ID: {validated_design.metadata.project_id}")
            print(f"👤 Owner: {validated_design.metadata.owner}")
            print(f"🔢 Qubits: {validated_design.quantum_design.qubit_count}")
            print(f"🎯 Frequency: {validated_design.quantum_design.target_frequency_ghz} GHz")
            print(f"🔗 Topology: {validated_design.quantum_design.topology}")
            
        except Exception as e:
            print(f"❌ Error loading design: {e}")
            current_design = None
            current_design_path = None

load_button.on_click(load_design)

# Display widgets
display(widgets.HBox([file_selector, load_button]))
display(output_area)

## Design Summary

Once a design is loaded, this section will display a comprehensive summary of the design parameters.

In [ ]:
# Design summary display
summary_output = widgets.Output()

def display_design_summary():
    with summary_output:
        summary_output.clear_output()
        
        if current_design is None:
            print("📝 No design loaded. Please load a design first.")
            return
        
        # Create summary markdown
        summary_md = f"""
        # Design Summary: {current_design.metadata.project_id}
        
        **Owner**: {current_design.metadata.owner}  
        **Created**: {current_design.metadata.created}  
        **Description**: {getattr(current_design.metadata, 'description', 'N/A')}
        
        ## Quantum Design
        - **Qubit Count**: {current_design.quantum_design.qubit_count}
        - **Topology**: {current_design.quantum_design.topology}
        - **Qubit Type**: {current_design.quantum_design.qubit_type}
        - **Target Frequency**: {current_design.quantum_design.target_frequency_ghz} GHz
        - **Anharmonicity**: {getattr(current_design.quantum_design, 'anharmonicity_mhz', 'N/A')} MHz
        - **T1 Target**: {getattr(current_design.quantum_design, 't1_target_microseconds', 'N/A')} μs
        - **T2 Target**: {getattr(current_design.quantum_design, 't2_target_microseconds', 'N/A')} μs
        
        ## Control System
        - **Backend**: {current_design.control.backend}
        - **Readout Resonance**: {current_design.control.readout_resonance_ghz} GHz
        - **Control Channels**: {current_design.control.control_channels}
        - **Readout Channels**: {getattr(current_design.control, 'readout_channels', 'N/A')}
        - **Sample Rate**: {getattr(current_design.control, 'sample_rate_gsps', 'N/A')} GS/s
        
        ## Packaging
        - **Package Type**: {current_design.packaging.package_type}
        - **Material**: {current_design.packaging.material}
        - **Thermal Conductivity**: {current_design.packaging.thermal_conductivity_w_m_k} W/m·K
        - **Die Size**: {getattr(current_design.packaging, 'die_size_mm', 'N/A')} mm²
        - **Wire Bonds**: {getattr(current_design.packaging, 'wire_bond_count', 'N/A')}
        
        ## Fabrication
        - **Foundry**: {current_design.fab.foundry}
        - **Process Node**: {current_design.fab.process_node}
        - **Metal Layers**: {current_design.fab.metal_layers}
        - **Minimum Feature**: {getattr(current_design.fab, 'minimum_feature_nm', 'N/A')} nm
        
        ## Simulation Configuration
        - **Process Corners**: {', '.join(current_design.simulation.process_corners)}
        - **Temperature Points**: {', '.join([f'{t}K' for t in current_design.simulation.temperature_points_k])}
        - **Monte Carlo Samples**: {current_design.simulation.monte_carlo_samples}
        """
        
        display(Markdown(summary_md))

# Auto-display when design is loaded
display_design_summary()

## Topology Visualization

Visualize the qubit connectivity topology and physical layout.

In [ ]:
# Topology visualization
viz_output = widgets.Output()

def visualize_topology():
    with viz_output:
        viz_output.clear_output()
        
        if current_design is None:
            print("📊 No design loaded. Please load a design first.")
            return
        
        # Create topology graph
        G = nx.Graph()
        
        # Add nodes (qubits)
        qubit_count = current_design.quantum_design.qubit_count
        for i in range(qubit_count):
            G.add_node(i, label=f'Q{i}')
        
        # Add edges based on topology
        topology = current_design.quantum_design.topology
        
        if topology == 'linear_chain':
            for i in range(qubit_count - 1):
                G.add_edge(i, i + 1)
        elif topology == 'heavy_hex':
            # Simplified heavy hex connectivity
            edges = [(0,1), (1,2), (2,3), (3,4), (4,5), (5,0),  # Outer hexagon
                    (0,6), (1,7), (2,8), (3,9), (4,10), (5,11),  # Inner connections
                    (6,7), (7,8), (8,9), (9,10), (10,11), (11,6)]  # Inner hexagon
            for edge in edges[:qubit_count]:
                if edge[0] < qubit_count and edge[1] < qubit_count:
                    G.add_edge(edge[0], edge[1])
        elif topology == 'grid_2d':
            # 2D grid layout
            import math
            grid_size = int(math.sqrt(qubit_count))
            for i in range(grid_size):
                for j in range(grid_size):
                    node = i * grid_size + j
                    if node >= qubit_count:
                        continue
                    # Connect to right neighbor
                    if j < grid_size - 1 and (i * grid_size + j + 1) < qubit_count:
                        G.add_edge(node, i * grid_size + j + 1)
                    # Connect to bottom neighbor
                    if i < grid_size - 1 and ((i + 1) * grid_size + j) < qubit_count:
                        G.add_edge(node, (i + 1) * grid_size + j)
        
        # Create visualization
        plt.figure(figsize=(10, 8))
        
        # Choose layout based on topology
        if topology == 'linear_chain':
            pos = nx.spring_layout(G, seed=42)
        elif topology == 'grid_2d':
            pos = {}
            grid_size = int(math.sqrt(qubit_count))
            for i in range(qubit_count):
                row = i // grid_size
                col = i % grid_size
                pos[i] = (col, -row)  # Negative y for top-to-bottom layout
        else:
            pos = nx.spring_layout(G, seed=42)
        
        # Draw the graph
        nx.draw(G, pos, with_labels=True, node_color='lightblue', 
                node_size=800, font_size=16, font_weight='bold',
                edge_color='gray', width=2)
        
        plt.title(f'Qubit Topology: {topology.replace("_", " ").title()} ({qubit_count} Qubits)', 
                 fontsize=16, pad=20)
        plt.axis('equal')
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        # Print connectivity statistics
        print(f"\n📊 Topology Statistics:")
        print(f"   Nodes (Qubits): {G.number_of_nodes()}")
        print(f"   Edges (Couplings): {G.number_of_edges()}")
        print(f"   Average Degree: {2 * G.number_of_edges() / G.number_of_nodes():.2f}")
        
        # Check if connected
        is_connected = nx.is_connected(G)
        print(f"   Connected: {'✅ Yes' if is_connected else '❌ No'}")
        
        if is_connected:
            diameter = nx.diameter(G)
            print(f"   Diameter: {diameter} (max shortest path)")

# Auto-visualize when design is loaded
visualize_topology()

## Parameter Analysis

Analyze key design parameters and their relationships.

In [ ]:
# Parameter analysis
param_output = widgets.Output()

def analyze_parameters():
    with param_output:
        param_output.clear_output()
        
        if current_design is None:
            print("📈 No design loaded. Please load a design first.")
            return
        
        # Extract key parameters
        freq = current_design.quantum_design.target_frequency_ghz
        readout_freq = current_design.control.readout_resonance_ghz
        qubit_count = current_design.quantum_design.qubit_count
        
        # Create parameter comparison plot
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(12, 10))
        
        # Frequency comparison
        ax1.bar(['Qubit', 'Readout'], [freq, readout_freq], 
                color=['blue', 'red'], alpha=0.7)
        ax1.set_ylabel('Frequency (GHz)')
        ax1.set_title('Operating Frequencies')
        ax1.grid(True, alpha=0.3)
        
        # Add frequency separation annotation
        separation = abs(readout_freq - freq)
        ax1.annotate(f'Δf = {separation:.1f} GHz', 
                    xy=(0.5, (freq + readout_freq)/2), 
                    xytext=(0.5, (freq + readout_freq)/2 + 0.1),
                    ha='center', fontsize=10,
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))
        
        # Control channels vs qubits
        control_channels = current_design.control.control_channels
        readout_channels = getattr(current_design.control, 'readout_channels', qubit_count)
        
        ax2.bar(['Control', 'Readout', 'Qubits'], 
                [control_channels, readout_channels, qubit_count],
                color=['green', 'orange', 'purple'], alpha=0.7)
        ax2.set_ylabel('Channel Count')
        ax2.set_title('Channel Requirements')
        ax2.grid(True, alpha=0.3)
        
        # Thermal properties
        thermal_cond = current_design.packaging.thermal_conductivity_w_m_k
        ax3.bar(['Thermal Conductivity'], [thermal_cond], 
                color=['cyan'], alpha=0.7)
        ax3.set_ylabel('W/m·K')
        ax3.set_title('Packaging Properties')
        ax3.grid(True, alpha=0.3)
        
        # Process corners
        corners = current_design.simulation.process_corners
        corner_values = {'tt': 0, 'ff': -1, 'ss': 1, 'fnsp': -0.5, 'snfp': 0.5}
        corner_positions = [corner_values.get(c.lower(), 0) for c in corners]
        
        ax4.scatter(range(len(corners)), corner_positions, s=100, alpha=0.7)
        ax4.set_xticks(range(len(corners)))
        ax4.set_xticklabels(corners, rotation=45)
        ax4.set_ylabel('Process Corner')
        ax4.set_title('Simulation Corners')
        ax4.grid(True, alpha=0.3)
        ax4.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        
        plt.tight_layout()
        plt.show()
        
        # Print parameter insights
        print("\n🔍 Parameter Insights:")
        
        # Frequency separation check
        if separation < 0.3:
            print("⚠️  WARNING: Qubit-readout frequency separation is tight (< 300 MHz)")
        elif separation > 1.0:
            print("ℹ️  NOTE: Large frequency separation may impact readout efficiency")
        else:
            print("✅ Good frequency separation for qubit-readout isolation")
        
        # Channel utilization
        control_util = control_channels / qubit_count
        readout_util = readout_channels / qubit_count
        print(f"📊 Control channel utilization: {control_util:.1f} channels/qubit")
        print(f"📊 Readout channel utilization: {readout_util:.1f} channels/qubit")
        
        # Thermal assessment
        if thermal_cond > 200:
            print("✅ Excellent thermal conductivity for cryogenic operation")
        elif thermal_cond > 150:
            print("👍 Good thermal conductivity")
        else:
            print("⚠️  Thermal conductivity may limit cooling performance")
        
        # Process corners
        print(f"🎲 Process corners to simulate: {', '.join(corners)}")
        if len(corners) >= 3:
            print("✅ Comprehensive corner analysis")
        else:
            print("ℹ️  Consider adding more process corners for robustness")

# Auto-analyze when design is loaded
analyze_parameters()

## Basic Validation Check

Run a quick validation check to ensure the design meets basic requirements.

In [ ]:
# Basic validation
validation_output = widgets.Output()
validate_button = widgets.Button(description='Run Basic Validation', button_style='success')

def run_basic_validation(b):
    with validation_output:
        validation_output.clear_output()
        
        if current_design is None:
            print("❌ No design loaded. Please load a design first.")
            return
        
        print("🔍 Running basic validation checks...")
        
        # Basic validation checks
        checks_passed = 0
        total_checks = 0
        
        # Check 1: Qubit count reasonable
        total_checks += 1
        if 1 <= current_design.quantum_design.qubit_count <= 100:
            print("✅ Qubit count is reasonable")
            checks_passed += 1
        else:
            print("❌ Qubit count seems unreasonable")
        
        # Check 2: Frequency in valid range
        total_checks += 1
        freq = current_design.quantum_design.target_frequency_ghz
        if 3.0 <= freq <= 10.0:
            print("✅ Qubit frequency in valid range")
            checks_passed += 1
        else:
            print(f"⚠️  Qubit frequency {freq} GHz is outside typical range (3-10 GHz)")
        
        # Check 3: Readout frequency separation
        total_checks += 1
        readout_freq = current_design.control.readout_resonance_ghz
        separation = abs(readout_freq - freq)
        if separation >= 0.2:
            print("✅ Sufficient qubit-readout frequency separation")
            checks_passed += 1
        else:
            print(f"❌ Insufficient frequency separation: {separation:.1f} GHz")
        
        # Check 4: Control channels adequate
        total_checks += 1
        control_channels = current_design.control.control_channels
        if control_channels >= current_design.quantum_design.qubit_count:
            print("✅ Adequate control channels")
            checks_passed += 1
        else:
            print(f"⚠️  Control channels ({control_channels}) may be insufficient for {current_design.quantum_design.qubit_count} qubits")
        
        # Check 5: Thermal conductivity reasonable
        total_checks += 1
        thermal_cond = current_design.packaging.thermal_conductivity_w_m_k
        if 100 <= thermal_cond <= 400:
            print("✅ Thermal conductivity in reasonable range")
            checks_passed += 1
        else:
            print(f"⚠️  Thermal conductivity {thermal_cond} W/m·K seems unusual")
        
        # Check 6: Process corners defined
        total_checks += 1
        corners = current_design.simulation.process_corners
        if len(corners) >= 1:
            print("✅ Process corners defined")
            checks_passed += 1
        else:
            print("❌ No process corners defined")
        
        # Summary
        print(f"\n📊 Validation Summary: {checks_passed}/{total_checks} checks passed")
        
        if checks_passed == total_checks:
            print("🎉 All basic validation checks passed!")
            print("\n💡 Next steps:")
            print("   1. Run full validation: qasic-validate design.json --full-stack")
            print("   2. Explore parameter sweeps in notebook 02")
            print("   3. Check yield analysis in notebook 03")
        elif checks_passed >= total_checks * 0.8:
            print("👍 Most checks passed. Review warnings above.")
        else:
            print("⚠️  Multiple issues found. Please review and fix.")

validate_button.on_click(run_basic_validation)

# Display validation interface
display(validate_button)
display(validation_output)

## Export Design Summary

Export the design summary and analysis to various formats.

In [ ]:
# Export functionality
export_output = widgets.Output()
export_button = widgets.Button(description='Export Summary', button_style='info')
export_format = widgets.Dropdown(
    options=['markdown', 'html', 'json'],
    value='markdown',
    description='Format:',
    style={'description_width': 'initial'}
)

def export_summary(b):
    with export_output:
        export_output.clear_output()
        
        if current_design is None:
            print("❌ No design loaded. Please load a design first.")
            return
        
        # Generate export content
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        project_id = current_design.metadata.project_id
        
        if export_format.value == 'markdown':
            content = f"""# Design Summary: {project_id}
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Design Parameters
- **Qubits**: {current_design.quantum_design.qubit_count}
- **Topology**: {current_design.quantum_design.topology}
- **Frequency**: {current_design.quantum_design.target_frequency_ghz} GHz
- **Control**: {current_design.control.backend}
- **Packaging**: {current_design.packaging.material}

## Validation Status
Basic validation checks completed. Run full validation for detailed analysis.
"""
            
            filename = f"{project_id}_summary_{timestamp}.md"
            
        elif export_format.value == 'html':
            content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Design Summary: {project_id}</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; }}
        h1 {{ color: #2E86AB; }}
        .summary {{ background: #f5f5f5; padding: 20px; border-radius: 5px; }}
    </style>
</head>
<body>
    <h1>Design Summary: {project_id}</h1>
    <p><strong>Generated:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    
    <div class="summary">
        <h2>Design Parameters</h2>
        <ul>
            <li><strong>Qubits:</strong> {current_design.quantum_design.qubit_count}</li>
            <li><strong>Topology:</strong> {current_design.quantum_design.topology}</li>
            <li><strong>Frequency:</strong> {current_design.quantum_design.target_frequency_ghz} GHz</li>
            <li><strong>Control:</strong> {current_design.control.backend}</li>
            <li><strong>Packaging:</strong> {current_design.packaging.material}</li>
        </ul>
        
        <h2>Validation Status</h2>
        <p>Basic validation checks completed. Run full validation for detailed analysis.</p>
    </div>
</body>
</html>"""
            
            filename = f"{project_id}_summary_{timestamp}.html"
            
        elif export_format.value == 'json':
            export_data = {
                'project_id': project_id,
                'generated': datetime.now().isoformat(),
                'design_summary': {
                    'qubit_count': current_design.quantum_design.qubit_count,
                    'topology': current_design.quantum_design.topology,
                    'frequency_ghz': current_design.quantum_design.target_frequency_ghz,
                    'control_backend': current_design.control.backend,
                    'packaging_material': current_design.packaging.material
                },
                'validation_status': 'basic_checks_completed'
            }
            content = json.dumps(export_data, indent=2)
            filename = f"{project_id}_summary_{timestamp}.json"
        
        # Write file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(content)
        
        print(f"✅ Summary exported to: {filename}")
        print(f"📄 Format: {export_format.value.upper()}")

export_button.on_click(export_summary)

# Display export interface
display(widgets.HBox([export_format, export_button]))
display(export_output)

## Next Steps

Now that you've explored your design, here are the recommended next steps:

1. **Full Validation**: Run the complete digital twin validation pipeline
   ```bash
   qasic-validate design.json --full-stack --workers 4
   ```

2. **Parameter Optimization**: Use notebook 02 for automated parameter sweeps

3. **Yield Analysis**: Check manufacturing yield predictions in notebook 03

4. **Thermal Analysis**: Review cooling performance in notebook 04

5. **Tapeout Preparation**: Final validation and packaging in notebook 05

---

**QASIC Digital Twin Platform** | *Accelerating quantum ASIC development from weeks to hours*